# Notebook 00 — Setup Colab Environment

## Goal
This notebook prepares the full environment for running the `economic-news-labor-rag` project in Google Colab.

**What this notebook does:**
1. Mounts Google Drive
2. Clones the GitHub repository
3. Installs all required Python packages
4. Creates the full folder structure in Google Drive
5. Checks that required API keys are set
6. Verifies that config files are present

**Run this notebook first, every time you start a new Colab session.**

---

## What can go wrong
- Google Drive not mounted → all subsequent notebooks will fail
- `pip install` fails → dependency conflict; check the error message
- `FRED_API_KEY` missing → Notebooks 01 and 03 will fail
- `DEEPSEEK_API_KEY` missing → Notebook 12 will use template fallback (not fatal)
- Git clone fails → check repo URL and internet connection

---

## What you must inspect
- ✅ Drive mount confirms `/content/drive/MyDrive` is accessible
- ✅ All required packages installed without errors
- ✅ All Drive folders created
- ✅ `FRED_API_KEY` found
- ✅ All 8 config files present

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Step 2 — Clone the repository and install requirements

In [ ]:


# Replace with your GitHub token if you need to authenticate from this notebook.
# Do not commit a real token to the repository.

# Git_key = 'your_github_token_here'



In [ ]:
import os

REPO_URL = 'https://github.com/mimimission/economic-news-labor-rag.git'  # UPDATE THIS
REPO_DIR = '/content/economic-news-labor-rag'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists at {REPO_DIR}, pulling latest...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -r requirements.txt -q
print('All packages installed.')

## Step 3 — Define Drive paths

In [ ]:
import pathlib

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')

FOLDERS = [
    DRIVE_ROOT / 'data' / 'raw' / 'fred',
    DRIVE_ROOT / 'data' / 'raw' / 'alfred',
    DRIVE_ROOT / 'data' / 'raw' / 'gdelt',
    DRIVE_ROOT / 'data' / 'interim' / 'macro',
    DRIVE_ROOT / 'data' / 'interim' / 'news',
    DRIVE_ROOT / 'data' / 'features' / 'macro',
    DRIVE_ROOT / 'data' / 'features' / 'news',
    DRIVE_ROOT / 'data' / 'model_ready',
    DRIVE_ROOT / 'artifacts' / 'models',
    DRIVE_ROOT / 'artifacts' / 'evidence_index',
    DRIVE_ROOT / 'outputs' / 'predictions',
    DRIVE_ROOT / 'outputs' / 'metrics',
    DRIVE_ROOT / 'outputs' / 'figures',
    DRIVE_ROOT / 'outputs' / 'tables',
    DRIVE_ROOT / 'outputs' / 'explanations',
    DRIVE_ROOT / 'outputs' / 'logs',
    DRIVE_ROOT / 'outputs' / 'data_quality',
    DRIVE_ROOT / 'approvals',
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Created {len(FOLDERS)} folders under {DRIVE_ROOT}')

## Step 4 — Verify folder structure

In [ ]:
import pandas as pd

rows = []
for folder in FOLDERS:
    rows.append({'path': str(folder.relative_to(DRIVE_ROOT)), 'exists': folder.exists()})

folder_df = pd.DataFrame(rows)
print(folder_df.to_string(index=False))

missing = folder_df[~folder_df['exists']]
if len(missing) > 0:
    print(f'\nWARNING: {len(missing)} folders were not created!')
    print(missing)
else:
    print('\nAll folders created successfully.')

## Step 5 — Check API keys

In [ ]:
import os

# --- SET YOUR API KEYS HERE ---
# Option A: set them directly (do not commit this cell with keys filled in)
# os.environ['FRED_API_KEY'] = 'your_fred_api_key'
# os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_api_key'

# Uncomment and fill in your keys locally if needed:
# os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_api_key'
# os.environ['FRED_API_KEY'] = 'your_fred_api_key'


# Option B: use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY')
# os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')

FRED_API_KEY = os.getenv('FRED_API_KEY')
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')

print('=== API Key Status ===')
if FRED_API_KEY:
    print(f'FRED_API_KEY:     FOUND ({FRED_API_KEY[:6]}...)')
else:
    print('FRED_API_KEY:     MISSING')
    print()
    print('STOP: FRED_API_KEY is required for FRED/ALFRED.')
    print('Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html')
    print('Set it with: os.environ["FRED_API_KEY"] = "your_key"')

if DEEPSEEK_API_KEY:
    print(f'DEEPSEEK_API_KEY: FOUND ({DEEPSEEK_API_KEY[:6]}...)')
else:
    print('DEEPSEEK_API_KEY: MISSING')
    print('WARNING: DeepSeek explanations will use template fallback (Notebook 12).')
    print('Forecasting and RAG retrieval will still work without this key.')


## Step 6 — Verify config files

In [ ]:
import pathlib

REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')
CONFIGS_DIR = REPO_DIR / 'configs'

expected_configs = [
    'base.yaml',
    'fred_series.yaml',
    'gdelt_queries.yaml',
    'cleaning_rules_news.yaml',
    'cleaning_rules_macro.yaml',
    'model_config.yaml',
    'evidence_config.yaml',
    'data_contracts.yaml',
]

print('=== Config File Check ===')
all_ok = True
for cfg in expected_configs:
    path = CONFIGS_DIR / cfg
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  {cfg:35s} {status}')
    if not path.exists():
        all_ok = False

if all_ok:
    print('\nAll config files present.')
else:
    print('\nWARNING: Some config files are missing. Check that the repo was cloned correctly.')

## Step 7 — Load and display base config

In [ ]:
import os

# --- SET YOUR API KEYS HERE ---
# Option A: set them directly (do not commit this cell with keys filled in)
# os.environ['FRED_API_KEY'] = 'your_fred_api_key'
# os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_api_key'

# Uncomment and fill in your keys locally if needed:
# os.environ['DEEPSEEK_API_KEY'] = 'your_deepseek_api_key'
# os.environ['FRED_API_KEY'] = 'your_fred_api_key'


# Option B: use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY')
# os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')

FRED_API_KEY = os.getenv('FRED_API_KEY')
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')

print('=== API Key Status ===')
if FRED_API_KEY:
    print(f'FRED_API_KEY:     FOUND ({FRED_API_KEY[:6]}...)')
else:
    print('FRED_API_KEY:     MISSING')
    print()
    print('STOP: FRED_API_KEY is required for FRED/ALFRED.')
    print('Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html')
    print('Set it with: os.environ["FRED_API_KEY"] = "your_key"')

if DEEPSEEK_API_KEY:
    print(f'DEEPSEEK_API_KEY: FOUND ({DEEPSEEK_API_KEY[:6]}...)')
else:
    print('DEEPSEEK_API_KEY: MISSING')
    print('WARNING: DeepSeek explanations will use template fallback (Notebook 12).')
    print('Forecasting and RAG retrieval will still work without this key.')

## Step 8 — Verify package imports

In [ ]:
import importlib

packages = [
    'pandas', 'numpy', 'pyarrow', 'requests', 'yaml', 'tqdm',
    'sklearn', 'xgboost', 'statsmodels', 'scipy', 'matplotlib',
    'joblib', 'openai'
]

print('=== Package Import Check ===')
all_ok = True
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'n/a')
        print(f'  {pkg:15s} {version}')
    except ImportError as e:
        print(f'  {pkg:15s} MISSING ({e})')
        all_ok = False

if all_ok:
    print('\nAll packages available.')
else:
    print('\nWARNING: Some packages failed to import. Re-run !pip install -r requirements.txt')

## Setup Summary

Before running Notebook 01, confirm:

| Check | Status |
|-------|--------|
| Google Drive mounted | ✅ / ❌ |
| Repo cloned | ✅ / ❌ |
| Packages installed | ✅ / ❌ |
| Drive folders created | ✅ / ❌ |
| `FRED_API_KEY` set | ✅ / ❌ |
| Config files present | ✅ / ❌ |

If all checks pass, proceed to `01_collect_raw_fred_alfred.ipynb`.